# Glüten — MedGemma 1.5 4B Marsh classifier (Unsloth QLoRA)

**Task:** Fine-tune `google/medgemma-1.5-4b` on IBDColEpi HE patches to classify a proxy Marsh grade.

**Honest framing — read this first.** IBDColEpi ships with pixel-level epithelium segmentation masks but **no Marsh / villous-atrophy / IEL annotations**. The labels trained below are weak-supervision proxies derived from epithelium mask coverage per patch (lower coverage ≈ more atrophy). They are methodologically principled but are **not pathologist-validated Marsh scores**. The hackathon writeup and the `/api/medgemma/marsh` endpoint surface this caveat explicitly.

**Target prize track:** Unsloth special technology.

**Runtime:** Kaggle free T4 (~3–4 h in fp32). Enable *Accelerator → GPU T4* and *Internet → On*.

**Attached data:** the private Kaggle dataset `gluten-ibdcolepi-sample` (uploaded from `data/structural/processed/patch-dataset-HE-sampled.zip` + `marsh_pseudo_labels-sampled.csv`).

**Setup quirks already accounted for in this notebook (don't undo them):**
- Pillow is left at Kaggle's pre-installed 12.2.0 — Unsloth installed with `--no-deps` so it can't downgrade Pillow and break the `_imaging` C extension.
- Vision tower is **frozen** (`finetune_vision_layers=False`). T4 can't train MedGemma's bf16 SigLIP encoder; freezing it means we only LoRA-tune the language head, which is all we need to teach the model the four Marsh labels.
- Trainer runs in fp32 (`fp16=False, bf16=False`) — slower but the only mode T4 + MedGemma both support.

## 1 · Install Unsloth + deps

All `--no-deps` so we don't fight Kaggle's base Pillow / transformers / datasets versions. If this cell errors, do **Run → Factory reset** and rerun from cell 1.

In [ ]:
%%capture
!pip install -q --no-deps "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install -q --no-deps cut-cross-entropy
!pip install -q tifffile

import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
import PIL; print('Pillow:', PIL.__version__)  # should print 12.2.0

## 2 · Build pseudo-Marsh labels from epithelium masks

For each HE patch we compute `epi_frac = (mask > 0).mean()`, then quantile-bin into Marsh-0 / Marsh-1 / Marsh-3a / Marsh-3b. Marsh-2 is omitted (under-represented in literature; molecular layer dataset GSE164883 also skips it). ~30 s on Kaggle disk.

In [ ]:
import pandas as pd, numpy as np
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

DATA_ROOT = Path('/kaggle/input/datasets/faithogun/gluten-ibdcolepi-sample')
PATCHES = DATA_ROOT / 'patch-dataset-HE-sampled'
SEED_CSV = DATA_ROOT / 'marsh_pseudo_labels-sampled.csv'

seed = pd.read_csv(SEED_CSV)
print(seed.shape, seed['split'].value_counts())

def read_mask(label_path):
    return np.array(Image.open(PATCHES / label_path))
def read_image(image_path):
    return Image.open(PATCHES / image_path).convert('RGB')

seed['epi_frac'] = [float((read_mask(r['label_path']) > 0).mean())
                    for _, r in tqdm(seed.iterrows(), total=len(seed))]

q = seed['epi_frac'].quantile([0.25, 0.5, 0.75]).values
def bin_fn(f):
    if f >= q[2]: return 'Marsh-0'
    if f >= q[1]: return 'Marsh-1'
    if f >= q[0]: return 'Marsh-3a'
    return 'Marsh-3b'
seed['marsh_bin'] = seed['epi_frac'].apply(bin_fn)
print(seed['marsh_bin'].value_counts())
seed.to_csv('/kaggle/working/marsh_pseudo_labels_scored.csv', index=False)

## 3 · Load MedGemma 1.5 4B in 4-bit, attach LoRA to the language head only

`finetune_vision_layers=False` is the load-bearing line. T4 can't backprop through MedGemma's bf16 SigLIP vision encoder, so we freeze it. We only LoRA-train the language part — that's enough to teach the model to output Marsh-0/1/3a/3b given the visual features SigLIP already extracts.

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    'unsloth/medgemma-1.5-4b-it-bnb-4bit',
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,   # frozen — see cell intro
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias='none', random_state=42,
)

## 4 · Build the image→label instruction dataset

In [ ]:
from datasets import Dataset

INSTRUCTION = (
    'You are a histopathology assistant. Classify the Marsh grade of this HE-stained '
    'intestinal biopsy patch. Respond with exactly one of: Marsh-0, Marsh-1, Marsh-3a, Marsh-3b.'
)

def load_patch(image_path):
    return read_image(image_path).resize((448, 448))

def to_example(row):
    img = load_patch(row['image_path'])
    return {
        'messages': [
            {'role': 'user', 'content': [
                {'type': 'image', 'image': img},
                {'type': 'text', 'text': INSTRUCTION},
            ]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': row['marsh_bin']}]},
        ]
    }

train_df = seed[seed['split'] == 'Trainset'].sample(min(2000, (seed['split'] == 'Trainset').sum()), random_state=42)
val_df = seed[seed['split'] == 'Validationset']

train_ds = Dataset.from_list([to_example(r) for _, r in train_df.iterrows()])
val_ds = Dataset.from_list([to_example(r) for _, r in val_df.iterrows()])
print(train_ds, val_ds)

## 5 · Train

fp32 mode (no fp16, no bf16). T4 + MedGemma forces this. Expect ~3–4 h for 250 steps. If it OOMs, drop `per_device_train_batch_size` to 1 and bump `gradient_accumulation_steps` to 8.

**Don't sit and wait** — use *Save Version → Save & Run All (Commit)* to run this in the background once the notebook is correct.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=False,
        bf16=False,
        logging_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='/kaggle/working/medgemma-marsh-lora',
        report_to='none',
        remove_unused_columns=False,
        dataset_text_field='',
        dataset_kwargs={'skip_prepare_dataset': True},
        max_length=2048,
    ),
)
trainer.train()

## 6 · Evaluate on the held-out Testset split

In [ ]:
FastVisionModel.for_inference(model)
from sklearn.metrics import classification_report, confusion_matrix

test_df = seed[seed['split'] == 'Testset'].sample(min(400, (seed['split'] == 'Testset').sum()), random_state=42)
preds, gold = [], []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    img = load_patch(row['image_path'])
    msgs = [{'role': 'user', 'content': [
        {'type': 'image', 'image': img},
        {'type': 'text', 'text': INSTRUCTION}]}]
    inp = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=inp, max_new_tokens=8, temperature=0, do_sample=False)
    txt = tokenizer.decode(out[0, inp.shape[-1]:], skip_special_tokens=True).strip()
    preds.append(txt.split()[0] if txt else '')
    gold.append(row['marsh_bin'])

print(classification_report(gold, preds, labels=['Marsh-0','Marsh-1','Marsh-3a','Marsh-3b']))
print(confusion_matrix(gold, preds, labels=['Marsh-0','Marsh-1','Marsh-3a','Marsh-3b']))

## 7 · Save LoRA adapter

~100 MB. Download from Kaggle's file pane (right side, /kaggle/working/medgemma-marsh-lora/). Then locally we wire it into the `/api/medgemma/marsh` route.

In [ ]:
model.save_pretrained('/kaggle/working/medgemma-marsh-lora')
tokenizer.save_pretrained('/kaggle/working/medgemma-marsh-lora')
!ls -lh /kaggle/working/medgemma-marsh-lora/